# Phase 0 — Kiểm tra môi trường Colab (MMDetection + ConvNeXt + Cascade R-CNN)

**Mục đích:** chứng minh *cả stack chạy được* **trước khi** viết bất kỳ dòng code
xử lý dữ liệu nào. Cụ thể gồm 4 việc:

1. Mount Google Drive và kiểm tra đường dẫn dữ liệu.
2. In ra môi trường runtime thật (Python, PyTorch, CUDA, GPU, dung lượng đĩa).
3. Cài `mmcv + mmengine + mmdet + mmpretrain` khớp đúng runtime.
4. **Chạy thử suy luận 1 ảnh** bằng đúng họ model của đồ án
   (Cascade Mask R-CNN + ConvNeXt-Tiny, pretrain COCO).

## Thứ tự chạy

| Ô | Việc | Ghi chú |
|---|---|---|
| [0.2] | Khai báo phiên bản + đường dẫn | sửa ở đây nếu cần |
| [0.3] | Mount Drive | |
| [0.4] | In môi trường runtime | |
| [0.6] | **Cài đặt** | chạy 1 lần, khoảng 3–5 phút |
| [0.7] | Kiểm chứng | import + chạy CUDA op thật |
| [0.9] | Smoke test suy luận | lần đầu tải checkpoint 344 MB |
| [0.10] | **BÁO CÁO GATE 0** | copy khối này gửi lại |

> ⚠️ **Không cần restart runtime.** Notebook chỉ *thêm* 4 gói mới, không thay
> PyTorch của Colab. Lý do ở ô [0.5].

> ⚠️ Trước khi chạy: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# ============================================================================
# [0.2] CẤU HÌNH — SỬA Ở ĐÂY
# ----------------------------------------------------------------------------
# Đây là chỗ DUY NHẤT chứa phiên bản thư viện và đường dẫn; các ô sau đọc lại từ
# đây. Khi cần đổi thì đổi một chỗ, không phải đi tìm khắp notebook.
# ============================================================================

# --- Nguồn wheel mmcv --------------------------------------------------------
# KHÔNG phải PyPI. OpenMMLab ngừng phát hành mmcv từ 7/2024, wheel chính thức
# chỉ có tới Python 3.12, mà Colab chạy Python 3.13 -> phải dùng index cộng đồng.
MMCV_INDEX = "https://miropsota.github.io/torch_packages_builder"

# --- Phiên bản cài từ PyPI ---------------------------------------------------
# Tra từ nguồn chính thức ngày 30/09/2026, xem docs/NOTES.md mục 1.
MMDET_VERSION       = "3.3.0"    # bản cuối của MMDetection (5/2024)
MMENGINE_VERSION    = "0.10.7"   # đã sửa lỗi trùng tên Adafactor (0.10.4 còn lỗi)
MMPRETRAIN_VERSION  = "1.2.0"    # BẮT BUỘC: backbone ConvNeXt nằm trong gói này
PYCOCOTOOLS_VERSION = "2.0.11"

# --- Repo MMDetection cần clone ---------------------------------------------
# Gói pip mmdet CÓ chứa config (đã mở thật wheel mmdet-3.3.0: có
# `mmdet/.mim/configs/`, 1.564 mục). Lý do phải clone nằm ở chỗ khác:
# `Config.fromfile` của mmengine đổi đường dẫn thành tuyệt đối rồi
# `check_file_exist` NGAY Ở DÒNG ĐẦU, mà cú pháp `mmdet::<đường dẫn>` chỉ
# được xử lý khi nằm trong danh sách `_base_` — không mở trực tiếp được
# một tệp config cấp cao nhất. Ô smoke test bên dưới cần mở trực tiếp
# một tệp như vậy, nên clone nông đúng tag là cách gọn nhất (~63 MB).
# (Config đồ án KHÔNG cần clone: nó dùng `mmdet::` trong `_base_`.)
MMDET_TAG  = "v3.3.0"
MMDET_GIT  = "https://github.com/open-mmlab/mmdetection.git"
MMDET_REPO = "/content/mmdetection"

# --- Đường dẫn dữ liệu -------------------------------------------------------
DRIVE_DIR   = "/content/drive/MyDrive/Flood_House_AI"
DRIVE_ZIP   = f"{DRIVE_DIR}/floodnet_raw.zip"   # dataset gốc (~12 GB, đã có sẵn)
RUNS_DIR    = f"{DRIVE_DIR}/runs"               # kết quả, checkpoint
WHEEL_CACHE = f"{DRIVE_DIR}/wheels"             # cache wheel mmcv (~55 MB)

# File đánh dấu đã cài xong -> chạy lại notebook trong cùng phiên sẽ không cài lại.
INSTALL_MARKER = "/content/.floodcount_env_ready"

# --- Model dùng cho smoke test ----------------------------------------------
# Đúng họ model của đồ án: Cascade + ConvNeXt-Tiny đã train trên COCO 80 lớp.
# LƯU Ý: model này KHÔNG biết lớp nhà ngập — đây chỉ là phép thử xem stack có
# chạy được không. Model thật của đồ án sẽ được train ở Phase 4.
SMOKE_CFG = (f"{MMDET_REPO}/configs/convnext/"
             "cascade-mask-rcnn_convnext-t-p4-w7_fpn_4conv1fc-giou_amp-ms-crop-3x_coco.py")
SMOKE_CKPT = ("https://download.openmmlab.com/mmdetection/v2.0/convnext/"
              "cascade_mask_rcnn_convnext-t_p4_w7_fpn_giou_4conv1f_fp16_ms-crop_3x_coco/"
              "cascade_mask_rcnn_convnext-t_p4_w7_fpn_giou_4conv1f_fp16_ms-crop_3x_coco_"
              "20220509_204200-8f07c40b.pth")
# Phương án nhẹ hơn nếu mạng chậm (192 MB thay vì 344 MB): đổi SMOKE_CKPT thành
# biến dưới đây là xong (model vẫn là ConvNeXt-Tiny, chỉ khác không có 3 tầng cascade).
SMOKE_CKPT_NHE = ("https://download.openmmlab.com/mmdetection/v2.0/convnext/"
                  "mask_rcnn_convnext-t_p4_w7_fpn_fp16_ms-crop_3x_coco/"
                  "mask_rcnn_convnext-t_p4_w7_fpn_fp16_ms-crop_3x_coco_"
                  "20220426_154953-050731f4.pth")

print("Da nap cau hinh Phase 0.")
print(f"  Drive       : {DRIVE_DIR}")
print(f"  Dataset zip : {DRIVE_ZIP}")
print(f"  Ket qua     : {RUNS_DIR}")
print(f"  Repo mmdet  : {MMDET_REPO} (tag {MMDET_TAG})")

In [ ]:
# ============================================================================
# [0.3] MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------------
# Dataset và thư mục kết quả đều nằm trên Drive. /content chỉ là đĩa tạm và bị
# xoá sạch khi hết phiên Colab, nên checkpoint BẮT BUỘC phải ghi ra Drive.
# ============================================================================
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================================
# [0.4] MÔI TRƯỜNG RUNTIME ĐANG CÓ (trước khi cài)
# ----------------------------------------------------------------------------
# In ra để đối chiếu khi có sự cố, và để biết Colab đang cấp phiên bản gì.
# Ô [0.6] không đọc các giá trị này — nó tự dò lại từ runtime — nhưng khi cài
# lỗi thì đây là chỗ đầu tiên cần xem.
# ============================================================================
import sys, subprocess
import torch


def lenh(cmd):
    """Chạy một lệnh hệ thống, trả về stdout.

    Trả về câu thông báo thay vì làm sập cả ô nếu runtime không có lệnh đó:
    runtime CPU-only không có `nvidia-smi`, và lỗi FileNotFoundError ném ra ở
    giữa ô sẽ che mất phần đĩa và RAM phía sau — những thứ vẫn in được bình
    thường và rất cần khi đi tìm nguyên nhân sự cố.
    """
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout
    except FileNotFoundError:
        return f"(runtime này không có lệnh `{cmd[0]}`)"


print("=" * 70)
print("MOI TRUONG RUNTIME HIEN TAI")
print("=" * 70)
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA (torch) : {torch.version.cuda}")
print(f"GPU kha dung : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {props.total_memory / 1e9:.1f} GB")
else:
    print()
    print("[!] CHUA BAT GPU. Vao Runtime > Change runtime type > chon T4 GPU,")
    print("    roi chay lai tu o [0.3]. Khong co GPU thi khong train duoc.")

print("\n--- nvidia-smi ---")
print(lenh(["nvidia-smi"]).strip() or "(nvidia-smi khong in ra gi)")

print("--- dung luong dia tam /content ---")
print(lenh(["df", "-h", "/content"]).strip())

print("--- RAM ---")
print(lenh(["free", "-h"]).strip())

## Ô [0.6] — Cài đặt, và 5 cái bẫy đã kiểm chứng

Ô này cài 4 gói và **không đụng vào PyTorch của Colab** (nên không cần restart):

1. **Dò tổ hợp Python / torch / CUDA của runtime** rồi tìm wheel `mmcv` khớp.
   Không hard-code tên file — Colab nâng cấp torch là tên đổi theo.
2. Cài `mmcv` bản **có CUDA ops** (không phải `mmcv-lite`). Lấy đúng wheel thì
   xong trong ~1 phút; lấy sai thì pip quay ra tải bản source và tự biên dịch —
   mất 30–60 phút và thường hỏng.
3. Cài `mmengine`, `mmdet`, `mmpretrain`, `pycocotools` từ PyPI.
4. Vá `mmdet/__init__.py`: mmdet 3.3.0 assert `mmcv < 2.2.0` (dấu `<` **nghiêm
   ngặt**), nên cài đúng bản 2.2.0 vẫn trượt. mmcv 2.2.0 chỉ là bản sửa lỗi so
   với 2.1.0, API không đổi → nới ngưỡng lên 2.3.0 là an toàn.
5. Tắt nhánh "đa phương thức" của mmpretrain (BLIP, BLIP2, LLaVA, OFA...) —
   xem bẫy thứ năm trong bảng dưới.

**Năm cái bẫy đã kiểm chứng bằng cách đọc source (xem `docs/NOTES.md` mục 1.2):**

| Bẫy | Xử lý trong notebook |
|---|---|
| mmcv không có wheel chính thức cho Python 3.13 | dò wheel cộng đồng theo đúng runtime |
| mmdet assert `mmcv < 2.2.0` là nghiêm ngặt | hàm `va_mmdet()` nới lên 2.3.0 |
| mmengine 0.10.4 trùng tên `Adafactor` (torch vs transformers) → `KeyError` | ghim 0.10.7 (đã sửa ở upstream); vẫn giữ hàm vá tự phát hiện |
| thiếu `mmpretrain` là chết ngay (config ConvNeXt đặt `allow_failed_imports=False`) | cài `mmpretrain` và kiểm chứng ở ô [0.7] |
| mmpretrain 1.2.0 viết cho transformers 4.x, Colab có 5.x → `import mmpretrain.models` nổ `TypeError: NoneType takes no arguments` | tắt cờ `WITH_MULTIMODAL` của chính OpenMMLab |

> **Không cài `opencv-python-headless`.** `mmengine` đã kéo theo `opencv-python`,
> hai gói này cùng cung cấp module `cv2` và cùng ghi vào một thư mục — cài chồng
> lên nhau sẽ làm hỏng `import cv2`.

Nếu cài thất bại và cần chạy lại: xoá `/content/.floodcount_env_ready` rồi chạy lại ô [0.6].

In [ ]:
# ============================================================================
# [0.6] CÀI ĐẶT MMCV + MMENGINE + MMDET + MMPRETRAIN
# ----------------------------------------------------------------------------
# Cài 4 gói rồi vá 2 chỗ. Xem giải thích ở ô markdown ngay trên.
# KHÁC cách làm quen thuộc: KHÔNG hạ PyTorch và KHÔNG cần restart runtime.
# ============================================================================
import os, re, sys, glob, subprocess, urllib.request, importlib.util, pathlib

# --- Kiểm tra ô [0.2] đã chạy chưa ------------------------------------------
# Các ô trong Jupyter dùng chung một kernel. Sau khi restart runtime, kernel mất
# hết biến cho tới khi chạy lại [0.2]. Dùng globals() để báo lỗi dễ hiểu thay vì
# NameError khó đoán.
if "INSTALL_MARKER" not in globals():
    raise SystemExit(
        "Ô [0.2] chưa được chạy trong kernel hiện tại.\n"
        "=> Chạy ô [0.2] (khai báo phiên bản + đường dẫn) TRƯỚC, rồi chạy lại ô này."
    )


def run(cmd):
    """Chạy lệnh shell, in ra lệnh trước khi chạy, dừng hẳn nếu thất bại."""
    print("\n$ " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {r.returncode}):\n    {cmd}")


def va_mmdet():
    """Nới ngưỡng mmcv tối đa mà mmdet chấp nhận: 2.2.0 -> 2.3.0.

    mmdet 3.3.0 assert `mmcv_version < digit_version('2.2.0')` — dấu nhỏ hơn
    NGHIÊM NGẶT, nên cài đúng bản 2.2.0 vẫn trượt assert. mmcv 2.2.0 chỉ là bản
    sửa lỗi so với 2.1.0, API không đổi, nên nới lên 2.3.0 là an toàn.

    Dùng find_spec thay vì import: import mmdet lúc này sẽ nổ ngay vì assert.
    """
    spec = importlib.util.find_spec("mmdet")
    if spec is None or spec.origin is None:
        return "KHÔNG thấy gói mmdet (cài đặt chưa xong?) — chạy lại ô này"
    f = pathlib.Path(spec.origin).parent / "__init__.py"
    s = f.read_text(encoding="utf-8")
    if "mmcv_maximum_version = '2.3.0'" in s:
        return "đã vá trước đó"
    s2 = s.replace("mmcv_maximum_version = '2.2.0'",
                   "mmcv_maximum_version = '2.3.0'")
    if s2 == s:
        return f"KHÔNG tìm thấy dòng cần vá — mở file vá tay: {f}"
    f.write_text(s2, encoding="utf-8")
    return f"đã vá ({f})"


def va_mmengine_neu_can():
    """Vá lỗi trùng tên `Adafactor` — CHỈ khi mmengine còn bản lỗi.

    Bản 0.10.4 đăng ký `Adafactor` từ HAI nguồn dưới cùng một tên: torch (vòng
    lặp `dir(torch.optim)`) và transformers. Năm 2024 torch chưa có lớp này nên
    không đụng nhau; torch 2.5+ đã thêm `torch.optim.Adafactor` và Colab cài sẵn
    `transformers` -> KeyError ngay khi import.
    Bản 0.10.7 đã sửa ở upstream (đăng ký bản của torch thành 'TorchAdafactor'),
    nên hàm này thường không làm gì. Giữ lại để nếu pip kéo về bản cũ thì tự vá.

    Lưu ý khi vá: dòng `OPTIMIZERS.register_module(module=_optim)` xuất hiện ở
    BỐN hàm khác nhau trong file, phải theo dõi đang ở hàm nào mới vá đúng chỗ.
    """
    spec = importlib.util.find_spec("mmengine")
    if spec is None or spec.origin is None:
        return "KHÔNG thấy gói mmengine (cài đặt chưa xong?) — chạy lại ô này"
    f = pathlib.Path(spec.origin).parent / "optim" / "optimizer" / "builder.py"
    s = f.read_text(encoding="utf-8")
    if "name='TorchAdafactor'" in s:
        return "không cần (mmengine đã sửa ở upstream)"
    if "if module_name in OPTIMIZERS:" in s:
        return "đã vá trước đó"

    dong = s.splitlines(keepends=True)
    ra, ham, i, so_cho = [], None, 0, 0
    while i < len(dong):
        l = dong[i]
        eol = "\r\n" if l.endswith("\r\n") else "\n"
        than = l.strip()
        thut = l[:len(l) - len(l.lstrip())]
        if than.startswith("def "):
            ham = than[4:].split("(")[0]

        # (1) Vòng lặp đăng ký optimizer của torch: bỏ qua tên đã có.
        if (ham == "register_torch_optimizers"
                and than == "OPTIMIZERS.register_module(module=_optim)"):
            ra.append(thut + "if module_name in OPTIMIZERS:" + eol)
            ra.append(thut + "    continue" + eol)
            ra.append(l)
            i += 1
            so_cho += 1
            continue

        # (2) transformers.Adafactor trùng tên với torch.optim.Adafactor.
        if (ham == "register_transformers_optimizers"
                and than == "OPTIMIZERS.register_module(name='Adafactor', module=Adafactor)"):
            ra.append(thut + "if 'Adafactor' not in OPTIMIZERS:" + eol)
            ra.append(thut + "    " + than + eol)
            if (i + 1 < len(dong) and dong[i + 1].strip()
                    == "transformer_optimizers.append('Adafactor')"):
                ra.append(thut + "    transformer_optimizers.append('Adafactor')" + eol)
                i += 2
            else:
                i += 1
            so_cho += 1
            continue

        ra.append(l)
        i += 1

    if not so_cho:
        return "KHÔNG vá được (không khớp mẫu) — báo lại để xem xét"
    f.write_text("".join(ra), encoding="utf-8")
    return f"đã vá {so_cho} chỗ ({f})"


def va_mmpretrain_da_phuong_thuc():
    """Tắt nhánh "đa phương thức" của mmpretrain — nếu không thì import nổ.

    Chuyện gì xảy ra: mmpretrain chỉ nạp BLIP / BLIP2 / LLaVA / OFA... khi cờ
    `WITH_MULTIMODAL = True`, mà cờ này bật khi máy có `transformers>=4.28.0`.
    Colab có transformers 5.x -> cờ bật -> nạp cả vườn model viết cho
    transformers 4.x -> nổ ngay:

        TypeError: NoneType takes no arguments

    Lỗi trên phát sinh trong `blip/language_model.py`: khối
    `try: from transformers.modeling_utils import ...` thất bại (transformers 5
    đã chuyển `apply_chunking_to_forward`, `prune_linear_layer` sang
    `pytorch_utils` và bỏ hẳn `find_pruneable_heads_and_indices`) nên nó gán
    `PreTrainedModel = None`, rồi VẪN đem `None` ra làm lớp cha. Cái `except:`
    trần của nó che mất nguyên nhân thật.

    Hậu quả không nhỏ: sập cả gói `mmpretrain.models`, kéo theo backbone
    ConvNeXt mà đồ án bắt buộc phải có (mmdet 3.3.0 không có ConvNeXt riêng).

    Đồ án chỉ dùng backbone ConvNeXt, không dùng model đa phương thức nào, nên
    cách sửa là tắt hẳn nhánh đó bằng ĐÚNG công tắc của OpenMMLab. Khi tắt,
    mmpretrain tự đăng ký bản thay thế cho các lớp đó — ai lỡ dùng sẽ nhận
    thông báo "cần cài mmpretrain[multimodal]" thay vì lỗi khó hiểu.
    """
    spec = importlib.util.find_spec("mmpretrain")
    if spec is None or spec.origin is None:
        return "KHÔNG thấy gói mmpretrain (cài đặt chưa xong?) — chạy lại ô này"

    f = pathlib.Path(spec.origin).parent / "utils" / "dependency.py"
    if not f.exists():
        return f"KHÔNG thấy file cần vá: {f}"
    s = f.read_text(encoding="utf-8")
    if "WITH_MULTIMODAL = False" in s:
        return "đã vá trước đó"

    cu = ("WITH_MULTIMODAL = all(\n"
          "    satisfy_requirement(item)\n"
          "    for item in ['pycocotools', 'transformers>=4.28.0'])")
    moi = ("# [floodcount] Tắt nhánh đa phương thức: nó viết cho transformers 4.x,\n"
           "# Colab dùng 5.x nên import nổ và làm sập cả gói mmpretrain.\n"
           "# Đồ án chỉ cần backbone ConvNeXt. Chi tiết: docs/NOTES.md muc 1.6.\n"
           "WITH_MULTIMODAL = False")
    s2 = s.replace(cu, moi)
    if s2 == s:
        return f"KHÔNG tìm thấy đoạn cần vá — mở file vá tay: {f}"
    f.write_text(s2, encoding="utf-8")
    return f"đã vá ({f})"


# ============================================================================
# PHẦN CÀI ĐẶT — chỉ chạy một lần
# ============================================================================
if os.path.exists(INSTALL_MARKER):
    print("[bỏ qua] Môi trường đã được cài ở lần chạy trước trong phiên này.")
    print("         Muốn cài lại từ đầu: xoá", INSTALL_MARKER, "rồi chạy lại ô này.")
else:
    import torch

    # --- Dò tổ hợp Python / torch / CUDA của runtime hiện tại ---------------
    PY_TAG = f"cp{sys.version_info.major}{sys.version_info.minor}"   # vd "cp313"
    TV = torch.__version__.split("+")[0]                             # vd "2.11.0"
    if torch.version.cuda is None:
        # Chặn sớm: bản torch CPU-only thì không có tổ hợp "cuXXX" nào để dò
        # wheel, và cũng không train được. Báo rõ cách sửa thay vì để pip lỗi.
        raise SystemExit(
            f"[!] Runtime này là bản CPU-only (torch {torch.__version__}) nên "
            "không có CUDA.\n"
            "=> Vào Runtime > Change runtime type > chọn T4 GPU, "
            "rồi chạy lại từ ô [0.3]."
        )
    CU = "cu" + torch.version.cuda.replace(".", "")                  # vd "cu128"
    print(f"Runtime: Python {sys.version.split()[0]} | torch {torch.__version__}")
    print(f"Cần wheel mmcv cho: {PY_TAG} + pt{TV} + {CU}")

    # --- Chọn chỗ cache wheel (ưu tiên Drive để phiên sau dùng lại) ---------
    if os.path.isdir(DRIVE_DIR):
        os.makedirs(WHEEL_CACHE, exist_ok=True)
    else:
        WHEEL_CACHE = "/content/wheels"
        os.makedirs(WHEEL_CACHE, exist_ok=True)
        print("[lưu ý] Chưa thấy thư mục Drive -> cache wheel vào /content/wheels")

    da_cache = glob.glob(f"{WHEEL_CACHE}/mmcv-*{PY_TAG}*pt{TV}{CU}*.whl")
    if da_cache:
        # Đã có wheel đúng tổ hợp trong cache -> cài thẳng, không phụ thuộc index.
        print(f"Dùng wheel đã cache:\n    {da_cache[0]}")
        run(f'pip install "{da_cache[0]}"')
    else:
        # --- Tìm wheel khớp trong index cộng đồng ---------------------------
        # Tên wheel có dạng:
        #   mmcv-2.2.0+a8073c7pt2.11.0cu128-cp313-cp313-linux_x86_64.whl
        # Chú ý: phải bắt phần CHỮ hiển thị của thẻ <a>, vì trong href dấu "+"
        # bị mã hoá thành "%2B" và regex sẽ khớp ra tên bị cắt cụt.
        with urllib.request.urlopen(f"{MMCV_INDEX}/mmcv/", timeout=60) as r:
            html = r.read().decode("utf-8", "replace")
        names = sorted(set(re.findall(r"mmcv-[0-9][-A-Za-z0-9._+]*\.whl", html)))
        match = [n for n in names
                 if PY_TAG in n and f"pt{TV}{CU}" in n and "linux_x86_64" in n]

        if not match:
            print(f"\n[!] KHÔNG có wheel mmcv khớp {PY_TAG} + pt{TV}{CU}.")
            print(f"    Các bản đang có cho {PY_TAG} (linux):")
            for n in names:
                if PY_TAG in n and "linux_x86_64" in n:
                    print("     ", n)
            raise SystemExit(
                "\n=> Nhiều khả năng Colab vừa nâng cấp torch/CUDA.\n"
                "   Gửi danh sách trên để cập nhật lại notebook."
            )

        wheel = match[0]
        # Phần phiên bản nằm giữa "mmcv-" và "-cp313-..."; pip cần đúng chuỗi này
        # (kể cả hậu tố +a8073c7pt2.11.0cu128) để chọn đúng bản dựng.
        mmcv_local = wheel[len("mmcv-"):].split(f"-{PY_TAG}")[0]
        print(f"Chọn wheel : {wheel}")
        print(f"Phiên bản  : mmcv=={mmcv_local}")

        # Phải là mmcv bản CÓ CUDA OPS (không phải mmcv-lite): mmdet dùng nms và
        # roi_align của mmcv trong cả lúc train lẫn lúc suy luận.
        run(f"pip install mmcv=={mmcv_local} --extra-index-url {MMCV_INDEX}")

        # Lưu wheel lên Drive để phiên sau cài từ cache — index cộng đồng có thể
        # đổi hoặc ngừng hoạt động, còn đồ án thì phải chạy lại được sau nhiều tháng.
        run(f'pip download mmcv=={mmcv_local} --no-deps '
            f'--extra-index-url {MMCV_INDEX} -d "{WHEEL_CACHE}"')

    # --- Ba gói còn lại từ PyPI ---------------------------------------------
    # mmdet KHÔNG khai báo mmcv/mmengine trong metadata (chỉ ở extra "mim") nên
    # pip sẽ không kéo mmcv về bản cũ — đã kiểm tra trên PyPI ngày 30/09/2026.
    run(f"pip install mmengine=={MMENGINE_VERSION} mmdet=={MMDET_VERSION} "
        f"mmpretrain=={MMPRETRAIN_VERSION} pycocotools=={PYCOCOTOOLS_VERSION}")

    with open(INSTALL_MARKER, "w") as f:
        f.write("ok")
    print("\n" + "!" * 72)
    print("CÀI ĐẶT XONG. Không cần restart runtime.")
    print("!" * 72)


# ============================================================================
# PHẦN VÁ — LUÔN chạy, kể cả khi đã cài từ phiên trước
# ----------------------------------------------------------------------------
# Đặt NGOÀI nhánh kiểm tra INSTALL_MARKER là cố ý: nếu để bên trong, một phiên
# đã cài xong nhưng chưa vá sẽ bỏ qua toàn bộ nhánh else và lỗi cũ quay lại.
# Các hàm vá đều tự kiểm tra xem đã vá chưa nên chạy lại nhiều lần vô hại.
# ============================================================================
print(f"\nvá mmdet      : {va_mmdet()}")
print(f"vá mmengine   : {va_mmengine_neu_can()}")
print(f"vá mmpretrain : {va_mmpretrain_da_phuong_thuc()}")

# ============================================================================
# XOÁ CACHE MODULE MMPRETRAIN — phải chạy ngay sau khi vá
# ----------------------------------------------------------------------------
# Python giữ các module đã nạp trong sys.modules. Nếu để nguyên, lần import sau
# sẽ dùng lại `mmpretrain.utils.dependency` đã nạp TRƯỚC khi vá — vẫn mang giá
# trị cờ CŨ — và việc vá coi như không có tác dụng. Xoá để nạp lại từ đĩa.
# ============================================================================
xoa = [t for t in list(sys.modules)
       if t == "mmpretrain" or t.startswith("mmpretrain.")]
for t in xoa:
    del sys.modules[t]
print(f"đã xoá {len(xoa)} module mmpretrain khỏi cache (sẽ nạp lại ở ô [0.7])")

In [ ]:
# ============================================================================
# [0.7] KIỂM CHỨNG MÔI TRƯỜNG
# ----------------------------------------------------------------------------
# Import suông KHÔNG đủ để biết mmcv dùng được: mmcv có thể import thành công
# nhưng các CUDA op lại hỏng. Vì vậy phần cuối ô này chạy thật một CUDA op
# (nms) trên GPU — đó mới là phép thử đúng. nms được dùng xuyên suốt cả lúc
# train lẫn lúc suy luận; nếu wheel thiếu kiến trúc GPU của T4 (sm_75) thì lỗi
# nổ ngay ở đây, tốt hơn nhiều so với việc nó nổ giữa lúc train.
# ============================================================================
import sys, importlib.metadata as md
import torch

if "INSTALL_MARKER" not in globals():
    raise SystemExit(
        "Ô [0.2] chưa được chạy trong kernel hiện tại.\n"
        "=> Chạy ô [0.2] TRƯỚC, rồi chạy lại ô này."
    )

print("=" * 70)
print("KIEM CHUNG MOI TRUONG")
print("=" * 70)
print(f"Python   : {sys.version.split()[0]}")
print(f"torch    : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda} | GPU kha dung: {torch.cuda.is_available()}")

import mmcv, mmengine, mmdet, mmpretrain

# KHÔNG dùng mmcv.__version__ để đoán bản dựng: mmcv/version.py ghi cứng
# __version__ = '2.2.0' nên luôn trả về đúng chuỗi đó, dù là wheel cộng đồng có
# CUDA ops hay bản tự biên dịch. Phiên bản THẬT nằm ở metadata của gói đã cài.
mmcv_meta = md.version("mmcv")
print(f"mmcv     : {mmcv_meta}   (mmcv.__version__ = {mmcv.__version__})")
print(f"mmengine : {mmengine.__version__}")
print(f"mmdet    : {mmdet.__version__}")
print(f"mmpretrain: {mmpretrain.__version__}")
# In ra để đối chiếu: chính transformers 5.x là thứ làm nhánh đa phương thức của
# mmpretrain nổ (bẫy thứ 5 ở ô [0.6]) — dù đồ án không hề dùng tới nó.
print(f"transformers: {md.version('transformers')}")

if "+" in mmcv_meta:
    print(f"  -> OK: bản dựng sẵn, khớp torch {torch.__version__}")
else:
    print("  -> [lưu ý] không có hậu tố '+': có thể là bản tự biên dịch từ source.")

# --- Phép thử THẬT: CUDA op của mmcv phải chạy được trên GPU ----------------
from mmcv.ops import nms
print("\nĐã import được mmcv.ops.nms (mmcv có CUDA ops, không phải mmcv-lite).")

if torch.cuda.is_available():
    boxes = torch.tensor([[0.0, 0.0, 10.0, 10.0],
                          [1.0, 1.0, 11.0, 11.0],
                          [50.0, 50.0, 60.0, 60.0]], device="cuda")
    scores = torch.tensor([0.9, 0.8, 0.7], device="cuda")
    keep, _ = nms(boxes, scores, iou_threshold=0.5)
    torch.cuda.synchronize()
    print(f"mmcv.ops.nms trên GPU: OK  (giữ lại {len(keep)}/3 box, kỳ vọng 2)")
else:
    print("[bỏ qua] Chưa bật GPU nên không chạy được phép thử CUDA op.")

# --- Backbone ConvNeXt: thứ mà config gốc của mmdet BẮT BUỘC phải có --------
# Config ConvNeXt đặt custom_imports với allow_failed_imports=False, thiếu gói
# là mmdet dừng ngay. Kiểm tra ở đây để phát hiện sớm.
from mmpretrain.models import ConvNeXt
print("mmpretrain.models.ConvNeXt: OK")

# Cờ đa phương thức phải TẮT — nếu BẬT thì phần vá ở ô [0.6] chưa có tác dụng.
from mmpretrain.utils.dependency import WITH_MULTIMODAL
print("Nhánh đa phương thức của mmpretrain: "
      + ("BẬT — quay lại chạy ô [0.6]!" if WITH_MULTIMODAL
         else "TẮT (đúng như đồ án cần)"))

from mmengine.registry import OPTIMIZERS
print(f"Optimizer đã đăng ký: TorchAdafactor={'TorchAdafactor' in OPTIMIZERS}, "
      f"Adafactor={'Adafactor' in OPTIMIZERS}")

print("\n=> MÔI TRƯỜNG SẴN SÀNG. Chuyển sang smoke test ở ô [0.9].")

## Ô [0.9] — Smoke test: chạy thử suy luận 1 ảnh

**Vì sao dùng đúng họ model của đồ án** (Cascade Mask R-CNN + ConvNeXt-Tiny)
mà không dùng một model đơn giản cho nhanh? Vì phép thử này phải chứng minh
đúng cái **đã từng chặn đồ án**: backbone ConvNeXt nằm trong `mmpretrain` có nạp
được không, và `mmcv` có chạy được roi_align/nms trong mạng thật không. Model
đơn giản hơn sẽ "chạy được" mà không chứng minh được điều gì.

**Vì sao phải clone repo mmdetection?** Gói pip của `mmdet` *có* chứa config
(đã mở thật wheel 3.3.0: `mmdet/.mim/configs/`, 1.564 mục), nhưng cú pháp
`mmdet::<đường dẫn>` chỉ dùng được khi nằm trong danh sách `_base_`: hàm
`Config.fromfile` đổi đường dẫn thành tuyệt đối rồi `check_file_exist` ngay
dòng đầu, nên **không** mở trực tiếp được một tệp config cấp cao nhất bằng
`mmdet::`. Ô smoke test cần mở trực tiếp một tệp như vậy, nên phải có cây
thư mục config thật -> clone nông đúng tag. (Config của đồ án thì **không**
cần clone: nó dùng `mmdet::` trong `_base_`.)

> ⚠️ Model tải về là model **COCO 80 lớp** — nó **không biết** lớp nhà ngập.
> Đây chỉ là phép thử stack. Model thật của đồ án sẽ được train ở Phase 4.

Kết quả mong đợi: ảnh `demo.jpg` được vẽ box với các lớp COCO (person, dog,
car...) và lưu vào `runs/phase0_smoke/vis/` trên Drive.

In [ ]:
# ============================================================================
# [0.9] SMOKE TEST — CHẠY THỬ SUY LUẬN 1 ẢNH
# ----------------------------------------------------------------------------
# Ba việc: clone repo mmdet (lấy config) -> nạp model pretrain -> suy luận 1 ảnh.
# Kết quả được lưu vào Drive để xem bằng mắt.
# ============================================================================
import os, time, subprocess
import torch

if "INSTALL_MARKER" not in globals():
    raise SystemExit("Chạy ô [0.2] trước.")

# --- Bước 1: clone repo mmdetection (chỉ lần đầu trong phiên) ----------------
if os.path.isdir(os.path.join(MMDET_REPO, ".git")):
    print(f"Đã có repo: {MMDET_REPO} -> bỏ qua clone.")
else:
    t0 = time.time()
    # --depth 1 --branch <tag>: chỉ lấy đúng bản cần, nhẹ hơn nhiều so với full.
    subprocess.run(["git", "clone", "--depth", "1", "--branch", MMDET_TAG,
                    MMDET_GIT, MMDET_REPO], check=True)
    print(f"Clone xong sau {time.time()-t0:.0f} giây.")

assert os.path.exists(SMOKE_CFG), f"Không thấy config: {SMOKE_CFG}"
IMG = os.path.join(MMDET_REPO, "demo", "demo.jpg")
assert os.path.exists(IMG), f"Không thấy ảnh demo: {IMG}"

# --- Bước 2: nạp model pretrain ---------------------------------------------
# Lần đầu sẽ tải checkpoint ~344 MB từ download.openmmlab.com.
from mmdet.apis import DetInferencer

print("Đang nạp model (lần đầu tải checkpoint ~344 MB)...")
t0 = time.time()
inferencer = DetInferencer(model=SMOKE_CFG, weights=SMOKE_CKPT, device="cuda:0")
print(f"Nạp xong sau {time.time()-t0:.0f} giây.")

# --- Bước 3: suy luận, lưu ảnh ----------------------------------------------
OUT_DIR = os.path.join(RUNS_DIR, "phase0_smoke")
os.makedirs(OUT_DIR, exist_ok=True)

t0 = time.time()
res = inferencer(IMG, out_dir=OUT_DIR, pred_score_thr=0.3, print_result=False)
torch.cuda.synchronize()
SMOKE_SECONDS = time.time() - t0

# --- Bước 4: in kết quả gọn (KHÔNG in cả dict: model có mask nên rất dài) ---
SMOKE_OK = "predictions" in res and len(res["predictions"]) > 0
classes = inferencer.model.dataset_meta["classes"]
pred0 = res["predictions"][0]
SMOKE_NBOX = len(pred0["labels"])

print(f"\nSuy luận xong sau {SMOKE_SECONDS:.2f} giây.")
print(f"Số box model trả về: {SMOKE_NBOX}")
print("\n10 box điểm cao nhất:")
for lbl, sc in sorted(zip(pred0["labels"], pred0["scores"]),
                      key=lambda t: -t[1])[:10]:
    print(f"   {classes[lbl]:<16} {sc:.3f}")

SMOKE_VIS = os.path.join(OUT_DIR, "vis")
print(f"\nẢnh kết quả lưu ở: {SMOKE_VIS}")
print("Mở thư mục này trên Drive để xem box có được vẽ đúng không.")

In [ ]:
# ============================================================================
# [0.10] BÁO CÁO GATE 0  ← copy khối in ra ở cuối ô này gửi lại
# ----------------------------------------------------------------------------
# Tổng hợp mọi thứ cần để chốt GATE 0: phiên bản thật của từng gói, kết quả
# kiểm chứng, kết quả smoke test, và tình trạng dataset trên Drive.
# Đồng thời ghi ra file trên Drive để sau này đối chiếu khi môi trường đổi.
# ============================================================================
import os, sys, datetime, subprocess, importlib.metadata as md
import torch

lines = []
def add(s=""):
    lines.append(s)
    print(s)

add("=" * 74)
add("BÁO CÁO GATE 0 — PHASE 0")
add("=" * 74)
add(f"Thời điểm chạy (giờ Colab): {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
add("")
add("--- MÔI TRƯỜNG ---")
add(f"Python        : {sys.version.split()[0]}")
add(f"PyTorch       : {torch.__version__}")
add(f"CUDA          : {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    add(f"GPU           : {torch.cuda.get_device_name(0)} ({p.total_memory/1e9:.1f} GB)")
else:
    add("GPU           : KHONG CO")
add("")
add("--- PHIÊN BẢN GÓI ĐÃ CÀI (đọc từ metadata, không phải từ biến __version__) ---")
for ten in ["mmcv", "mmengine", "mmdet", "mmpretrain", "pycocotools", "torch",
            "transformers", "numpy", "opencv-python"]:
    try:
        add(f"{ten:<14}: {md.version(ten)}")
    except Exception:
        add(f"{ten:<14}: (khong cai)")
add("")
add("--- KIỂM CHỨNG ---")
try:
    from mmcv.ops import nms
    add("mmcv.ops.nms  : import OK")
except Exception as e:
    add(f"mmcv.ops.nms  : LOI -> {e}")
try:
    add(f"mmcv co CUDA ops: {'+' in md.version('mmcv')}")
except Exception as e:
    add(f"mmcv metadata   : LOI -> {e}")
try:
    from mmpretrain.utils.dependency import WITH_MULTIMODAL
    add("mmpretrain đa phương thức: "
        + ("BẬT (sai — phần vá chưa có tác dụng)" if WITH_MULTIMODAL else "TẮT (đúng)"))
except Exception as e:
    add(f"mmpretrain đa phương thức: LOI -> {e}")
add("")
add("--- SMOKE TEST (Cascade Mask R-CNN + ConvNeXt-T, COCO) ---")
add(f"Chạy được     : {globals().get('SMOKE_OK', 'CHUA CHAY o [0.9]')}")
add(f"Số box        : {globals().get('SMOKE_NBOX', '-')}")
add(f"Thời gian     : {globals().get('SMOKE_SECONDS', 0):.2f} giây")
add(f"Ảnh kết quả   : {globals().get('SMOKE_VIS', '-')}")
add("")
add("--- DATASET ---")
if os.path.exists(DRIVE_ZIP):
    add(f"floodnet_raw.zip: {os.path.getsize(DRIVE_ZIP)/1e9:.2f} GB  ({DRIVE_ZIP})")
else:
    add(f"floodnet_raw.zip: KHONG THAY  ({DRIVE_ZIP})")
    add("  -> Kiểm tra lại tên thư mục / tên file trên Drive.")
add("")
add("--- DUNG LƯỢNG ĐĨA TẠM ---")
# Không dùng hàm `lenh` của ô [0.4]: ô này phải chạy được cả khi người dùng
# chạy riêng nó (kernel chưa có biến của các ô trước).
try:
    add(subprocess.run(["df", "-h", "/content"], capture_output=True,
                       text=True).stdout.strip())
except FileNotFoundError:
    add("(runtime này không có lệnh `df`)")
add("=" * 74)
add("KẾT THÚC BÁO CÁO")
add("=" * 74)

# Ghi ra Drive để lưu lại (nếu Drive đã mount).
thu_muc = RUNS_DIR if os.path.isdir(RUNS_DIR) else "/content"
os.makedirs(thu_muc, exist_ok=True)
duong_dan = os.path.join(thu_muc, "BAO_CAO_GATE0.txt")
with open(duong_dan, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print(f"\nĐã ghi báo cáo vào: {duong_dan}")